# Demand Forecasting V1 on Kaggle
Attach three Kaggle datasets to this notebook:
1. Code dataset
2. M5 dataset
3. Walmart dataset

In [ ]:
import os
from pathlib import Path

# Replace with your Kaggle dataset folder names under /kaggle/input
CODE_DATASET = 'your-code-dataset'
M5_DATASET = 'your-m5-dataset'
WALMART_DATASET = 'your-walmart-dataset'

PROJECT_DIR = Path('/kaggle/working/DS Demand Forecasting')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR

In [ ]:
def copy_or_unzip(dataset_name: str, expected_folder: str, zip_name: str):
    base = Path('/kaggle/input') / dataset_name
    folder_path = base / expected_folder
    zip_path = base / zip_name

    if folder_path.exists():
        os.system(f"cp -r '{folder_path}' '{PROJECT_DIR}'")
        return

    if zip_path.exists():
        os.system(f"unzip -o '{zip_path}' -d '{PROJECT_DIR}'")
        return

    raise FileNotFoundError(f'Could not find {expected_folder} or {zip_name} in {base}')

# Code dataset can be folder style or zip style
code_base = Path('/kaggle/input') / CODE_DATASET
code_zip = code_base / 'project_code.zip'
if code_zip.exists():
    os.system(f"unzip -o '{code_zip}' -d '{PROJECT_DIR}'")
else:
    os.system(f"cp -r '{code_base}'/* '{PROJECT_DIR}'")

copy_or_unzip(M5_DATASET, 'm5-forecasting-accuracy', 'm5_data.zip')
copy_or_unzip(WALMART_DATASET, 'archive', 'walmart_data.zip')

print('Project directory:', PROJECT_DIR)
print('Top-level files:')
for p in sorted(PROJECT_DIR.iterdir()):
    print('-', p.name)

In [ ]:
%cd '/kaggle/working/DS Demand Forecasting'
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt

## Run Data Pipeline
Build merged and feature engineered datasets in data/processed.

In [ ]:
!python scripts/run_data_pipeline.py

In [ ]:
# Fast baseline model
!python -c "from src.models.lightgbm import train_and_predict; train_and_predict()"

In [ ]:
# Optional heavier models
RUN_SARIMA = False
RUN_PROPHET = False

if RUN_SARIMA:
    os.system("python -c \"from src.models.sarima import train_and_predict; train_and_predict()\"")

if RUN_PROPHET:
    os.system("python -c \"from src.models.prophet import train_and_predict; train_and_predict()\"")

print('Optional model run complete')

In [ ]:
!python scripts/run_evaluation.py

import pandas as pd
metrics = pd.read_csv('outputs/metrics/model_comparison.csv')
impact = pd.read_csv('outputs/metrics/business_impact.csv')
display(metrics)
display(impact)

In [ ]:
!zip -r outputs_bundle.zip outputs data/predictions data/processed/split_metadata.json
print('Created outputs_bundle.zip in /kaggle/working')